# Preproccessing of the dataset

In [ ]:
import pandas as pd
import json
import os
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image


In [ ]:
train_data_path = '../raw_data/train/train_data/'
train_labels_path = '../raw_data/train/train_labels.json'
test_data_path = '../raw_data/test/test_data/'
test_labels_path = '../raw_data/test/test_labels.json'


In [ ]:
with open(train_labels_path) as f:
    train_data = json.load(f)

In [ ]:
train_images = pd.DataFrame(train_data['images'])
train_annotations = pd.DataFrame(train_data['annotations'])
train_categories = pd.DataFrame(train_data['categories'])

train_images['image_id'] = train_images['id']
train_images = train_images.drop(columns=['id'])

train_categories['category_name'] = train_categories['name']
train_categories = train_categories.drop(columns=['name'])

train_categories['category_id'] = train_categories['id']
train_categories = train_categories.drop(columns=['id'])

train_df = train_images.copy()

train_df = train_df.merge(
    train_annotations,
    on='image_id',
    how='inner'
    )
train_df['path'] = train_df['file_name'].apply(
    lambda name: os.path.join(train_data_path, name)
)

train_df = train_df.merge(
    train_categories,
    left_on='category_id',
    right_on='category_id',
    how='inner'
    )

train_df['egg_id'] = train_df['id']
train_df = train_df.drop(columns=['license', 'coco_url', 'id'])

train_df

In [ ]:
with open(test_labels_path) as f:
    test_data = json.load(f)

In [ ]:
test_images = pd.DataFrame(test_data['images'])
test_annotations = pd.DataFrame(test_data['annotations'])
test_categories = pd.DataFrame(test_data['categories'])

test_images['image_id'] = test_images['id']
test_images = test_images.drop(columns=['id'])

test_categories['category_name'] = test_categories['name']
test_categories = test_categories.drop(columns=['name'])

test_categories['category_id'] = test_categories['id']
test_categories = test_categories.drop(columns=['id'])

test_df = test_images.copy()

test_df = test_df.merge(
    test_annotations,
    on='image_id',
    how='inner'
    )
test_df['path'] = test_df['file_name'].apply(
    lambda name: os.path.join(test_data_path, name)
)

test_df = test_df.merge(
    test_categories,
    left_on='category_id',
    right_on='category_id',
    how='inner'
    )

test_df['egg_id'] = test_df['id']
test_df = test_df.drop(columns=['license', 'coco_url', 'id'])

test_df

In [ ]:
def show_with_box(row: pd.Series):
    img = Image.open(row['path'])
    x, y, w, h = row['bbox']

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(img)
    ax.add_patch(patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='blue', facecolor='none'))
    ax.axis('off')
    plt.show()

show_with_box(train_df.iloc[324])

In [ ]:
def crop_egg(row: pd.Series, margin: float = 0.2):
    img = Image.open(row['path'])
    x, y, w, h = row['bbox']

    # add some space around the egg (20% of the box size)
    mx, my = w * margin, h * margin
    left   = max(0, x - mx)
    top    = max(0, y - my)
    right  = min(img.width,  x + w + mx)
    bottom = min(img.height, y + h + my)

    return img.crop((left, top, right, bottom))

crop_egg(train_df.iloc[324])

### This code is how we get all image_ids that contain 2 or more eggs in one image.

  **`multiple_eggs = images_with_annotations.value_counts('image_id').sort_values(ascending=False).head(30)`**

In [ ]:
multiple_eggs = train_df.value_counts('image_id').sort_values(ascending=False).head(30)
multiple_eggs